# 14 · GSE232381 · bulk_RNA_seq · WGCNA modules (exploratory)

Reads `wgcna_input.rds`. Writes `modules.rds` and the end product `module_genes.csv` in
`data/run_artifacts/GSE232381/`.

Settings as for GSE65391 (notebook 05): signed network and TOM, minModuleSize 30, mergeCutHeight 0.25.
There are more genes than fit one block, so `maxBlockSize = 5000`: WGCNA pre-clusters the genes into
blocks and builds modules within each block.

In [1]:
if (!requireNamespace("WGCNA", quietly = TRUE)) install.packages("WGCNA", repos = "https://cloud.r-project.org")
suppressMessages(library(WGCNA))
source("../src/paths.R")
w <- readRDS(art("GSE232381", "wgcna_input.rds"))
c(samples = nrow(w$datExpr), genes = ncol(w$datExpr), power = w$power)

samples   genes   power 
     16   16605      10

In [2]:
set.seed(SEED)
net <- blockwiseModules(w$datExpr, power = w$power, networkType = "signed", TOMType = "signed",
                        minModuleSize = 30, mergeCutHeight = 0.25, maxBlockSize = 5000,
                        numericLabels = FALSE, pamRespectsDendro = FALSE, verbose = 0)
sizes <- sort(table(net$colors), decreasing = TRUE)
c(modules = sum(names(sizes) != "grey"), genes_in_modules = sum(sizes[names(sizes) != "grey"]), grey = unname(sizes["grey"]))
sizes

modules genes_in_modules             grey 
              34            16338              267


     turquoise           blue          brown         yellow          green 
          4293           3710           1720            796            571 
           red          black           pink        magenta         purple 
           530            429            367            345            327 
   greenyellow            tan           grey         salmon           cyan 
           270            269            267            246            241 
  midnightblue      lightcyan         grey60     lightgreen    lightyellow 
           232            219            182            175            163 
     royalblue        darkred      darkgreen  darkturquoise       darkgrey 
           149            146            137            136             97 
        orange     darkorange          white        skyblue    saddlebrown 
            89             77             66             58             57 
     steelblue  paleturquoise         violet darkolivegreen    darkmagenta 
           

**Result.** 34 modules holding 16,338 of 16,605 genes; 267 are grey. With 16 samples,
correlations between unrelated genes are large by chance, so almost every gene joins a module. This is
what the FAQ's warning about small samples predicts.

In [3]:
ME  <- orderMEs(net$MEs)
kME <- cor(w$datExpr, ME)
modules <- setdiff(names(sizes), "grey")
hubs <- lapply(setNames(modules, modules), function(mo) {
  g <- names(net$colors)[net$colors == mo]
  names(sort(kME[g, paste0("ME", mo)], decreasing = TRUE))
})
t(sapply(hubs, function(h) paste(head(h, 8), collapse = ", ")))

turquoise,blue,brown,yellow,green,red,black,pink,magenta,purple,⋯,orange,darkorange,white,skyblue,saddlebrown,steelblue,paleturquoise,violet,darkolivegreen,darkmagenta
"ZNF675, TMEM209, N4BP2, TADA1, LYRM7, ZBTB35, PHF6, SOX30-AS1","UBAP1, STXBP2, VSIR, ATP6AP1, FMNL1, VASP, ZYX, FCER1G","ARHGAP27P1-BPTFP1-KPNA2P3, SMIM48, KATNIP, TBC1D3L, PGAM2, MIR29B2CHG, RGL2, UNKL","NDUFS7, TMED9, CYC1, GSTP1, WDR46, EIF5A, SEC13, NDUFV1","RC3H2, EFR3A, IPO8, RAB8B, TBC1D23, PUM2, UHMK1, SCYL2","ISOC2, MRPL14, MRPS12, PRMT1, MRPS7, NDUFB2, NDUFAB1, HSPBP1","IGHV3-13, IGKV3D-15, IGLV3-10, DERL3, IGHV4-34, IGKV1D-33, IGLV1-51, IGKV1D-39","NPIPA7, CELF6, FOXH1, NSUN5P1, IL11RA, MAMDC4, METTL17, KLHL17","RNF111, NUFIP2, RLF, CREBRF, FHIP2A, LCOR, RALGAPB, ATF7IP","CCT6A, ABCF2, DDX1, GART, MESD, SNX5, POLR2D, NDUFA12",⋯,"KRABD4, SNTA1, PNPLA7, COQ10A, EPS8L2, BCAS4, RFX5-AS1, PMS2P1","IARS2, ADSS2, DIAPH2, LNPK, PAIP1, AP3M1, ESYT2, SHLD2P3","AZU1, CEACAM8, CEACAM6, PRTN3, MMP8, DEFA4, ELANE, BPI","RNA5S6, RNA5S11, RNA5S3, RNA5S15, RNA5S1, RNA5S14, RNA5S13, RNA5S17","AKAP1, METTL21C, CCDC120, TARBP1, ZNF121, ZNF337, TIMM21, FBXL19-AS1","ZBTB44, UBQLN1, TAX1BP1, GHITM, CCNY, ME2, RNF14, DYM","IFI44L, HERC6, USP18, CMPK2, SPATS2L, RSAD2, IFIT1, IFIT5","ARHGAP19-SLIT1, LINC00260, LOC107986710, LOC105373311, FTX, HIF1A-AS1, LOC105373835, SPDYE10","DCAF4L1, CDC42BPG, KRT72, DLEU2, RNFT2, LOC105373431, ZNF844, MARCHF3-DT","MSS51, PTMAP11, HM13-AS1, LINC00265, MIR570, TIRAP-AS1, IQCC, ERMN"


In [4]:
ifn6 <- read_gene_set("ifn-type1-6.txt")
data.frame(gene = ifn6, module = net$colors[ifn6], row.names = NULL)

gene,module
<chr>,<chr>
IFI27,cyan
IFI44L,paleturquoise
IFIT1,paleturquoise
ISG15,blue
RSAD2,paleturquoise
SIGLEC1,blue


**Result.** Three modules read clearly by their hub genes: paleturquoise (IFI44L, HERC6, USP18,
RSAD2, IFIT1: interferon), white (AZU1, CEACAM8, CEACAM6, PRTN3, ELANE: neutrophil granule) and black
(IGHV, IGKV, IGLV genes: immunoglobulin, plasma cell). The six interferon-score genes are split across
three modules here (cyan, paleturquoise, blue).

In [5]:
in_module <- names(net$colors)[net$colors != "grey"]
module_genes <- data.frame(gene = in_module, module = net$colors[in_module],
                           kME = kME[cbind(in_module, paste0("ME", net$colors[in_module]))], row.names = NULL)
write.csv(module_genes, art("GSE232381", "module_genes.csv"), row.names = FALSE)
saveRDS(list(colors = net$colors, MEs = ME, kME = kME, hubs = hubs), art("GSE232381", "modules.rds"))